In [13]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [15]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [16]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    
    return output

In [17]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
def grade_syntax(output, test_case):
    if test_case["format"] == "json":
        return validate_json(output)
    elif test_case["format"] == "python":
        return validate_python(output)
    elif test_case["format"] == "regex":
        return validate_regex(output)
    else:
        raise ValueError(f"Unknown expected output type: {test_case['expected_output_type']}")

In [18]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
def run_test_case(test_case): 
  output = run_prompt(test_case) 

  print(f"Output:\n{output}\n")

  # Grade the output 
  model_grade = grade_by_model(test_case, output) 
  model_score = model_grade["score"]

  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score) / 2
  reasoning = model_grade["reasoning"] 

  return { 
    "output": output, 
    "test_case": test_case, 
    "score": score, 
    "reasoning": reasoning 
  }

In [20]:
from statistics import mean 

def run_eval(dataset): 
  results = [] 

  for test_case in dataset: 
    result = run_test_case(test_case) 
    results.append(result) 

  average_score = mean([result["score"] for result in results]) 

  print(f"Average score: {average_score}") 

  return results

In [21]:
import json

with open("lesson11-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Output:

```python
def parse_arn(arn):
    parts = arn.split(':', 5)
    return {
        'partition': parts[1],
        'service': parts[2],
        'region': parts[3],
        'account-id': parts[4],
        'resource': parts[5]
    }
```
```

Output:
```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::my-data-bucket/*"
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:ListBucket"
      ],
      "Resource": "arn:aws:s3:::my-data-bucket"
    }
  ]
}
```

Output:
```python
import re

pattern = r'^sg-[0-9a-f]{8}$|^sg-[0-9a-f]{17}$'
```

Average score: 5.333333333333333


In [22]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n```python\ndef parse_arn(arn):\n    parts = arn.split(':', 5)\n    return {\n        'partition': parts[1],\n        'service': parts[2],\n        'region': parts[3],\n        'account-id': parts[4],\n        'resource': parts[5]\n    }\n```\n```",
    "test_case": {
      "task": "Write a Python function that parses an ARN (Amazon Resource Name) and returns a dictionary with its components: partition, service, region, account-id, and resource. The function should handle ARNs in the format 'arn:partition:service:region:account-id:resource'.",
      "format": "python"
    },
    "score": 3.0,
    "reasoning": "The solution demonstrates understanding of the ARN structure and correctly uses split with maxsplit to preserve colons in the resource component. However, it's a fragile implementation that assumes perfect input. Production code should validate that the string starts with 'arn:', has at least 6 components, and handle cases where region or account-id might be